# Anime Recommender — PySpark Setup
Java 17 is pre-installed in this container so no `--add-opens` workarounds are needed.

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('AnimeRecommender') \
    .config('spark.ui.port', '4040') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')
print('Spark UI available at http://localhost:4040')

Spark version: 3.5.0
Spark UI available at http://localhost:4040


In [2]:
# Paths inside the container (mapped from your ./data folder on the host)
BASE = '/home/jovyan/work/data/preprocessed'

reviews_data = spark.read.csv(f'{BASE}/reviews.csv',  header=True, inferSchema=True)
anime_data   = spark.read.csv(f'{BASE}/animes.csv',   header=True, inferSchema=True)

print('reviews schema:')
reviews_data.printSchema()
print('animes schema:')
anime_data.printSchema()

reviews schema:
root
 |-- _c0: integer (nullable = true)
 |-- uid: integer (nullable = true)
 |-- anime_uid: integer (nullable = true)
 |-- score: integer (nullable = true)

animes schema:
root
 |-- _c0: integer (nullable = true)
 |-- anime_id: integer (nullable = true)
 |-- genre: string (nullable = true)



In [3]:
# Index the string 'profile' column into a numeric user_id
indexer = StringIndexer(inputCol='profile', outputCol='user_id')
reviews_indexed = indexer.fit(reviews_data).transform(reviews_data)

final_data = reviews_indexed.select(
    col('user_id').cast('integer'),
    col('anime_uid').cast('integer'),
    col('score').cast('float')
)

final_data.show(7)

Py4JJavaError: An error occurred while calling o42.fit.
: org.apache.spark.SparkException: Input column profile does not exist.
	at org.apache.spark.ml.feature.StringIndexerBase.$anonfun$validateAndTransformSchema$2(StringIndexer.scala:128)
	at scala.collection.TraversableLike.$anonfun$flatMap$1(TraversableLike.scala:293)
	at scala.collection.IndexedSeqOptimized.foreach(IndexedSeqOptimized.scala:36)
	at scala.collection.IndexedSeqOptimized.foreach$(IndexedSeqOptimized.scala:33)
	at scala.collection.mutable.ArrayOps$ofRef.foreach(ArrayOps.scala:198)
	at scala.collection.TraversableLike.flatMap(TraversableLike.scala:293)
	at scala.collection.TraversableLike.flatMap$(TraversableLike.scala:290)
	at scala.collection.mutable.ArrayOps$ofRef.flatMap(ArrayOps.scala:198)
	at org.apache.spark.ml.feature.StringIndexerBase.validateAndTransformSchema(StringIndexer.scala:123)
	at org.apache.spark.ml.feature.StringIndexerBase.validateAndTransformSchema$(StringIndexer.scala:115)
	at org.apache.spark.ml